## Установка pyspark

In [1]:
!pip install pyspark --quiet
!pip install -U -q PyDrive --quiet
!apt install openjdk-8-jdk-headless &> /dev/null

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

!wget https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip &> /dev/null
!unzip ngrok-stable-linux-amd64.zip &> /dev/null
get_ipython().system_raw('./ngrok http 4050 &')


## Задача

В PySpark приложении датафреймами(pyspark.sql.DataFrame) заданы продукты, категории и их связи. Каждому продукту может соответствовать несколько категорий или ни одной. А каждой категории может соответствовать несколько продуктов или ни одного. Напишите метод на PySpark, который в одном датафрейме вернет все пары «Имя продукта – Имя категории» и имена всех продуктов, у которых нет категорий.

```
# Выбран кодовый формат
```



### Подготовка

In [2]:
from pyspark.sql import SparkSession

from pyspark.sql import DataFrame
from pyspark.sql.functions import col


def get_all_pairs(
        products: DataFrame,
        categories: DataFrame,
        product_category_relation: DataFrame) -> DataFrame:

    products_with_relations = products.join(product_category_relation, on="productId", how="left")
    products_with_categories = products_with_relations.join(categories, on="categoryId", how="left")

    return products_with_categories.select(col("productName"), col("categoryName"))



### Запуск


In [3]:
spark = SparkSession.builder.appName("ProductCategory").getOrCreate()

products_df = spark.createDataFrame([
    {"productId": 1, "productName": "Product A"},
    {"productId": 2, "productName": "Product B"},
    {"productId": 3, "productName": "Product C"},
    {"productId": 4, "productName": "Product D"},
])

categories_df = spark.createDataFrame([
    {"categoryId": 1, "categoryName": "Category X"},
    {"categoryId": 2, "categoryName": "Category Y"},
    {"categoryId": 3, "categoryName": "Category Z"},
])

product_categories_df = spark.createDataFrame([
    {"productId": 1, "categoryId": 1},
    {"productId": 1, "categoryId": 2},
    {"productId": 2, "categoryId": 2},
    {"productId": 3, "categoryId": 3},
])

final_df = get_all_pairs(products_df, categories_df, product_categories_df)
final_df.show()

+-----------+------------+
|productName|categoryName|
+-----------+------------+
|  Product A|  Category Y|
|  Product A|  Category X|
|  Product B|  Category Y|
|  Product C|  Category Z|
|  Product D|        NULL|
+-----------+------------+

